# Stabilized S-JEPA PreLocal

**SUPERSEDED STABILITY FRAMEWORK.** Later strict LP-FT improved stability but not accuracy and is stopped. Do not rerun or tune this older grid; see `AGENTS.md` section 2e. Execution fails closed.

# 1. Setup

In [ ]:
raise RuntimeError('CLOSED superseded stabilized-PreLocal framework: see AGENTS.md section 2e.')
import os, sys, json, glob, random, hashlib, builtins, platform, inspect
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import scipy.io as sio
from scipy import signal, stats
from sklearn.model_selection import StratifiedKFold, GroupKFold, LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.covariance import OAS
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix, f1_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
print(f"Python: {sys.version.split()[0]} | Platform: {platform.platform()} | cwd: {Path.cwd()}")

# 2. Configuration
## 2.1 Domain Defaults
## 2.2 CONFIG

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent
CONFIG = {
    # Paths / identity
    "artifact_dir":str(WORKING_DIR/"artifacts"/"liu2024-sjepa-stabilized-prelocal"), "source_extract_dir":str(WORKING_DIR/"liu2024_data"/"liu2024_figshare"/"sourcedata"),
    "experiment_name":"sjepa_stabilized_prelocal", "config_note":"Fold-safe PreLocal stabilization and frozen-probe controls.",
    # Dataset / preprocessing
    "subjects_to_use":[1,3,7,9,10,11,14,15,17,29,31,32,37,41], "sfreq":128, "window_start_s":0.0, "window_samples":512, "reference_mode":"average",
    # Model
    "pretrained_repo_id":"braindecode/signal-jepa_without-chans", "mode":"stabilized_prelocal", # stabilized_prelocal | frozen_probe
    "spatial_deviation_weight":0.01, "row_norm_weight":0.001, "orthogonality_weight":0.001, "project_row_norm":False,
    "gradient_clip_norm":1.0, "classifier_learning_rate":1e-3, "spatial_learning_rate":1e-4, "weight_decay":1e-4,
    # Evaluation/training: validation is split only from outer training.
    "cv_folds":5, "val_fraction":0.2, "max_epochs":200, "early_stopping_patience":20, "batch_size":8,
    "ensemble_seeds":[2026], "collapse_threshold":0.90,
    # Reproducibility
    "seed":2026, "set_seed":True,
}

## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    config_hash = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"
RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=False)
LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")
def _safe_write_text(stream, text):
    try: stream.write(text)
    except UnicodeEncodeError:
        enc = getattr(stream, "encoding", None) or "utf-8"
        stream.write(text.encode(enc, errors="replace").decode(enc, errors="replace"))
def _timestamped_print(*args, **kwargs):
    sep, end = kwargs.pop("sep", " "), kwargs.pop("end", "\n")
    flush, target = kwargs.pop("flush", False), kwargs.pop("file", None)
    message = sep.join(str(a) for a in args)
    stamped = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}" if message else ""
    for stream in ([target] if target is not None else [sys.stdout, _LOG_FILE_HANDLE]): _safe_write_text(stream, stamped + end)
    if flush: _LOG_FILE_HANDLE.flush()
builtins.print = _timestamped_print
config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f: json.dump(CONFIG, f, indent=2)
print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")

## 2.4 Reproducibility

In [ ]:
BASE_SEED = int(CONFIG["seed"])
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed); random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
        torch.use_deterministic_algorithms(True, warn_only=True)
    except ImportError: pass
if CONFIG["set_seed"]: seed_everything(BASE_SEED)
print(f"Seed initialized: {BASE_SEED}")

# 3. Load and Prepare Data
## 3.1 Data Loading Helpers

In [ ]:
EEG_IDX = [i for i in range(30) if i != 17]
CH_NAMES = ["Fp1","Fp2","Fz","F3","F4","F7","F8","FCz","FC3","FC4","FT7","FT8","Cz","C3","C4","T3","T4","CP3","CP4","TP7","TP8","Pz","P3","P4","T5","T6","Oz","O1","O2"]
def find_files(root):
    files = sorted(Path(root).glob("sub-*/sub-*_eeg.mat"))
    if not files: raise FileNotFoundError(root)
    return files
def load_subject(path):
    eeg = sio.loadmat(path)["eeg"][0, 0]
    raw = np.asarray(eeg["rawdata"], float); y = np.asarray(eeg["label"]).ravel().astype(int) - 1
    marker = raw[:, 32]; onsets=[]
    for m in marker:
        idx=np.flatnonzero(m == 2); valid=idx[(idx >= 800) & (idx <= 1300)]
        onsets.append(int(valid[0]) if len(valid) else 1003)
    sid=int(path.parent.name.split("-")[1])
    return sid, raw[:, EEG_IDX], y, np.asarray(onsets)
def selected_files():
    keep = CONFIG["subjects_to_use"]
    return [p for p in find_files(CONFIG["source_extract_dir"]) if keep is None or int(p.parent.name.split("-")[1]) in set(keep)]

## 3.2 Fold-Safe Windows

In [ ]:
import torch
from torch.utils.data import DataLoader,TensorDataset
from braindecode.models import SignalJEPA_PreLocal
DEVICE=torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
def windows(raw,onsets):
    out=[]
    for x,o in zip(raw,onsets):
        z=x[:,o:o+2000]
        if CONFIG["reference_mode"]=="average": z=z-z.mean(0,keepdims=True)
        elif CONFIG["reference_mode"]!="none": raise ValueError("reference_mode must be average or none")
        z=signal.resample_poly(z,CONFIG["sfreq"],500,axis=-1); out.append(z[:,:CONFIG["window_samples"]])
    return np.asarray(out,np.float32)
def build_model():
    model=SignalJEPA_PreLocal.from_pretrained(CONFIG["pretrained_repo_id"],n_chans=29,chs_info=None,n_times=CONFIG["window_samples"],n_outputs=2,strict=False).to(DEVICE)
    if CONFIG["mode"]=="frozen_probe":
        for p in model.parameters(): p.requires_grad=False
        for p in model.final_layer.parameters(): p.requires_grad=True
    return model
def spatial_layer(model):
    block=getattr(model,"spatial_conv",None)
    if block is None: raise AttributeError("SignalJEPA_PreLocal has no spatial_conv")
    candidates=[m for m in block.modules() if isinstance(m,torch.nn.Conv1d)]
    if not candidates: raise AttributeError("SignalJEPA_PreLocal spatial_conv contains no Conv1d")
    return candidates[0]
def train_fold(X,y,tr,te,seed):
    seed_everything(seed); inner=StratifiedKFold(5,shuffle=True,random_state=seed); fit_idx,val_idx=next(inner.split(X[tr],y[tr])); fit_idx,trval=tr[fit_idx],tr[val_idx]
    model=build_model(); spatial=spatial_layer(model); initial=spatial.weight.detach().clone()
    spatial_ids={id(p) for p in spatial.parameters()}; groups=[{"params":[p for p in model.parameters() if p.requires_grad and id(p) not in spatial_ids],"lr":CONFIG["classifier_learning_rate"]},{"params":[p for p in spatial.parameters() if p.requires_grad],"lr":CONFIG["spatial_learning_rate"]}]
    opt=torch.optim.AdamW(groups,weight_decay=CONFIG["weight_decay"]); loss_fn=torch.nn.CrossEntropyLoss(); best=None; best_loss=float("inf"); patience=0
    loader=DataLoader(TensorDataset(torch.from_numpy(X[fit_idx]),torch.from_numpy(y[fit_idx]).long()),batch_size=CONFIG["batch_size"],shuffle=True)
    for epoch in range(CONFIG["max_epochs"]):
        model.train()
        for xb,yb in loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); opt.zero_grad(); logits=model(xb); w=spatial.weight
            deviation=(w-initial).square().mean(); norms=w.flatten(1).norm(dim=1); row=(norms-1).square().mean(); gram=w.flatten(1)@w.flatten(1).T; orth=(gram-torch.diag(torch.diag(gram))).square().mean()
            loss=loss_fn(logits,yb)+CONFIG["spatial_deviation_weight"]*deviation+CONFIG["row_norm_weight"]*row+CONFIG["orthogonality_weight"]*orth; loss.backward()
            if CONFIG["gradient_clip_norm"]: torch.nn.utils.clip_grad_norm_(model.parameters(),CONFIG["gradient_clip_norm"])
            opt.step()
            if CONFIG["project_row_norm"]:
                with torch.no_grad(): spatial.weight.div_(spatial.weight.flatten(1).norm(dim=1).clamp_min(1e-8).view(-1,*([1]*(spatial.weight.ndim-1))))
        model.eval()
        with torch.no_grad(): vl=loss_fn(model(torch.from_numpy(X[trval]).to(DEVICE)),torch.from_numpy(y[trval]).long().to(DEVICE)).item()
        if vl<best_loss-1e-6: best_loss=vl; best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; patience=0
        else: patience+=1
        if patience>=CONFIG["early_stopping_patience"]: break
    model.load_state_dict(best); model.eval()
    with torch.no_grad(): logits=model(torch.from_numpy(X[te]).to(DEVICE)); prob=logits.softmax(1)[:,1].cpu().numpy(); pred=logits.argmax(1).cpu().numpy()
    hist=np.bincount(pred,minlength=2); collapse=float(hist.max()/hist.sum())>=CONFIG["collapse_threshold"]
    return pred,prob,{"epochs":epoch+1,"best_validation_loss":best_loss,"collapsed":collapse,"prediction_histogram":hist.tolist(),"spatial_deviation_l2":float((spatial.weight.detach().cpu()-initial.cpu()).norm())}

# 4. Model
## 4.1 Stabilized PreLocal and Frozen Probe

# 5. Training
## 5.1 Outer CV with Outer-Train Validation

In [ ]:
DATA={}
for p in selected_files():
    sid,raw,y,on=load_subject(p); DATA[sid]=(windows(raw,on),y)
SUBJECTS=sorted(DATA); FOLD_RESULTS=[]
for sid in SUBJECTS:
    X,y=DATA[sid]
    for fold,(tr,te) in enumerate(StratifiedKFold(CONFIG["cv_folds"],shuffle=True,random_state=BASE_SEED).split(X,y)):
        probs=[]; diagnostics=[]
        for seed in CONFIG["ensemble_seeds"]:
            pred,prob,diag=train_fold(X,y,tr,te,seed+fold); probs.append(prob); diagnostics.append(diag)
        score=np.mean(probs,axis=0); pred=(score>=0.5).astype(int); hist=np.bincount(pred,minlength=2)
        FOLD_RESULTS.append({"subject_id":sid,"fold_id":fold,"accuracy":accuracy_score(y[te],pred),"balanced_accuracy":balanced_accuracy_score(y[te],pred),"y_true":y[te].tolist(),"y_pred":pred.tolist(),"score":score.tolist(),"confusion_matrix":confusion_matrix(y[te],pred,labels=[0,1]).tolist(),"prediction_histogram":hist.tolist(),"collapsed":bool(hist.max()/hist.sum()>=CONFIG["collapse_threshold"]),"member_diagnostics":diagnostics})
SUBJECT_METRICS=[]
for sid in SUBJECTS:
    rr=[r for r in FOLD_RESULTS if r["subject_id"]==sid]; yt=np.concatenate([r["y_true"] for r in rr]); yp=np.concatenate([r["y_pred"] for r in rr]); sc=np.concatenate([r["score"] for r in rr])
    SUBJECT_METRICS.append({"subject_id":sid,"pooled_oof_accuracy":accuracy_score(yt,yp),"pooled_oof_balanced_accuracy":balanced_accuracy_score(yt,yp),"pooled_oof_auc":roc_auc_score(yt,sc),"collapse_rate":np.mean([r["collapsed"] for r in rr])})
GLOBAL_METRICS={"primary":"pooled_oof_per_subject_then_subject_mean","mean_balanced_accuracy":float(np.mean([r["pooled_oof_balanced_accuracy"] for r in SUBJECT_METRICS])),"mean_collapse_rate":float(np.mean([r["collapse_rate"] for r in SUBJECT_METRICS])),"outer_test_used_for_selection":False}

# 6. Results
## 6.1 Collapse Diagnostics and Artifact Saving

In [ ]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"
subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"
global_metrics_path = ARTIFACT_DIR / "global_metrics.json"
pd.DataFrame(FOLD_RESULTS).to_json(cv_results_path, orient="records", indent=2)
pd.DataFrame(SUBJECT_METRICS).to_json(subject_metrics_path, orient="records", indent=2)
with open(global_metrics_path, "w") as f: json.dump(GLOBAL_METRICS, f, indent=2)
run_metadata = {"run_id": RUN_ID, "artifact_dir": str(ARTIFACT_DIR), "experiment_name": CONFIG["experiment_name"], "config_note": CONFIG["config_note"], "subjects": [int(s) for s in SUBJECTS], "channel_names": CH_NAMES, "seed": BASE_SEED, "global_metrics": GLOBAL_METRICS, "performance_artifacts": {"cv_results": str(cv_results_path), "subject_metrics": str(subject_metrics_path), "global_metrics": str(global_metrics_path)}}
run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, "w") as f: json.dump(run_metadata, f, indent=2)
print(f"CV results saved to:      {cv_results_path}")
print(f"Subject metrics saved to: {subject_metrics_path}")
print(f"Global metrics saved to:  {global_metrics_path}")
print(f"Run metadata saved to:    {run_metadata_path}")
print(f"\nAll artifacts in: {ARTIFACT_DIR}")
try: _LOG_FILE_HANDLE.close()
except Exception: pass